In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import date_format, col
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import to_timestamp


# -----------------------------
# 1. Spark session
# -----------------------------
spark = SparkSession.builder.appName("SalesDataGeneration").getOrCreate()

# -----------------------------
# 2. Read customer IDs
# -----------------------------
customer_df = spark.sql("SELECT CustomerID FROM Oracle.DIM_CUSTOMERDATA")  
customer_ids = [row.CustomerID for row in customer_df.collect()]

# -----------------------------
# 3. Product master
# -----------------------------
product_master = [
    ("PROD0001", "Pale Meadow", "Paint Shades"),
    ("PROD0002", "Tranquil Lavender", "Paint Shades"),
    ("PROD0039", "Paint Safe Drop Cloth", "Paint Accessories"),
    ("WINE001", "Crimson Cabernet", "Beverages - Wine"),
    ("WINE003", "Velvet Merlot", "Beverages - Wine"),
    ("SOLAR001", "SunPower Panel X", "Solar"),
    ("APP001", "Winter Jacket", "Apparel"),
    ##("APP002", "Raincoat", "Apparel"),
    ("APP003", "Fleece Pullover", "Apparel"),
]

stores = ["Los Angeles", "Seattle", "Ontario", "San Francisco", "Mountain View"]
now = datetime.now()

# -----------------------------
# 4. Determine starting TransactionID
# -----------------------------
# Read last TransactionID from FactSales table
try:
    last_txn = spark.sql("SELECT max(TransactionID) as max_txn FROM Snowflake.FactSales").collect()[0]['max_txn']
    if last_txn:
        # Extract numeric part and increment
        last_num = int(last_txn.replace("TXN", ""))
    else:
        last_num = 0
except:
    last_num = 0

# -----------------------------
# 5. Generate 10 new sales records
# -----------------------------
sales_records = []
for i in range(100):
    prod = random.choice(product_master)
    store = random.choice(stores)
    quantity = random.randint(1, 5)
    price = round(random.uniform(10, 240), 2)
    #sale_date = now - timedelta(days=random.randint(0, 60), 
                             #   hours=random.randint(0, 23), 
                             #   minutes=random.randint(0,59))
    today = datetime.now().date()
    random_time = timedelta(
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    sale_date = datetime.combine(today, datetime.min.time()) + random_time


    customer_id = random.choice(customer_ids)
    
    sales_records.append({
        "TransactionID": f"TXN{last_num + i + 1:07d}",  # continue from last
        "StoreLocation": store,
        "ProductID": prod[0],
        "ProductName": prod[1],
        "ProductCategory": prod[2],
        "Quantity": quantity,
        "UnitPrice": price,
        "SaleDate": sale_date,
        "CUSTOMERID": customer_id
    })

# -----------------------------
# 6. Convert to Pandas DataFrame
# -----------------------------
df_sales = pd.DataFrame(sales_records)
df_sales = df_sales.dropna(subset=["CUSTOMERID"])
df_sales["CUSTOMERID"] = df_sales["CUSTOMERID"].astype(int)
df_sales["IngestionDate"] = datetime.now()  # Pandas column

# -----------------------------
# 7. Define schema
# -----------------------------
schema = StructType([
    StructField("TransactionID", StringType(), True),
    StructField("StoreLocation", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("ProductCategory", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("SaleDate", TimestampType(), True),
    StructField("CUSTOMERID", IntegerType(), True),
    StructField("IngestionDate", TimestampType(), True), 
])

# -----------------------------
# 8. Convert Pandas → Spark DataFrame
# -----------------------------
spark_df_sales = spark.createDataFrame(df_sales, schema=schema)

# Ensure SaleDate stays as Timestamp
spark_df_sales = spark_df_sales.withColumn("SaleDate", to_timestamp(col("SaleDate")))

spark_df_sales = spark_df_sales.withColumn(
    "IngestionDate", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

# -----------------------------
# 9. Append to Delta table
# -----------------------------
spark_df_sales.write.mode("append").format("delta").option("mergeSchema", "false").saveAsTable("Snowflake.factsales")


StatementMeta(, 2c100112-7ec9-49ff-9ad3-0a407700bb06, 3, Finished, Available, Finished)

In [ ]:
df = spark.sql("SELECT * FROM #LakehouseSilver#.Snowflake.factsales LIMIT 1000")
display(df)

StatementMeta(, 2c100112-7ec9-49ff-9ad3-0a407700bb06, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, db905379-44c5-4616-a24f-7ed99d61e6e2)